# Samatrix ResumeForge 2026: End-to-End Resume Classification System
**A Practical NLP Workflow: Problem ? Data ? EDA ? Preprocessing ? Features ? Classical ML ? Deep Learning ? Evaluation ? Error Analysis ? Demo**

---

### Project Objectives & Constraints
1. **Multi-Class Classification**: Predict the job category of a resume among **24 professional categories**.
2. **Domain-Aware NLP**: Specifically preserve critical technical tokens (e.g. C++, .NET, Python, SQL, AWS, TensorFlow, Java, NLP).
3. **Data Integrity & Leakage Prevention**: Stratified train/val/test splitting before any vectorization or vocabulary fitting.
4. **Comprehensive Benchmarking**: Compare classical ML baselines (*Multinomial Naive Bayes, Logistic Regression, Linear SVM*) against a neural deep learning approach (*Word2Vec + BiLSTM*).
5. **Rigorous Evaluation & Error Analysis**: Evaluate on Accuracy, Macro-F1, Weighted-F1, Precision, and Recall; inspect misclassified samples to understand real-world failure modes.


## Stage 1 ? Environment & Project Setup
We configure the project environment, import dependencies, and verify GPU/CPU availability.


In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add repository root to path
sys.path.insert(0, os.path.abspath('..'))

from src.pdf_extractor import extract_text_from_pdf, build_dataset_from_raw_pdfs
from src.preprocessing import clean_resume_text, assess_data_quality, extract_skills_from_text
from src.features import split_data, build_tfidf_features, train_word2vec_embeddings, build_embedding_matrix, texts_to_sequences
from src.models_classical import train_classical_models, evaluate_predictions
from src.models_dl import ResumeBiLSTM, train_dl_model
from src.evaluate import generate_comparison_table, plot_confusion_matrix, perform_error_analysis
from src.inference import ResumeClassifierPipeline

print(f"PyTorch Version: {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("All project modules successfully loaded.")


## Stage 2 ? PDF to Dataset Extraction
Every PDF across all 24 category folders in data/raw/ is parsed, extracting raw text, calculating document length, and associating the folder name as the target category.


In [ ]:
extracted_csv = os.path.join('..', 'data', 'processed', 'extracted_resumes.csv')
if not os.path.exists(extracted_csv):
    print("Extracting PDFs from data/raw...")
    df_raw = build_dataset_from_raw_pdfs(raw_dir='../data/raw', output_csv=extracted_csv)
else:
    df_raw = pd.read_csv(extracted_csv)

print(f"Total Extracted Resumes: {len(df_raw)}")
print(f"Unique Categories ({df_raw['category'].nunique()}): {sorted(df_raw['category'].unique())}")
df_raw[['id', 'category', 'char_count', 'word_count', 'is_empty']].head()


## Stage 3 ? Data Quality & Imbalance Audit
We assess:
- Missing and empty values
- Exact and near-duplicate resumes
- Short resumes (< 30 words)
- Class imbalance ratio across all 24 categories


In [ ]:
quality_report = assess_data_quality(df_raw)
print("=== DATA QUALITY REPORT ===")
for k, v in quality_report.items():
    if k != 'category_distribution':
        print(f"  {k:28s}: {v}")

print(f"\nClass Imbalance Ratio (Max:Min) = {quality_report['imbalance_ratio']}:1")


## Stage 4 ? Exploratory Data Analysis (EDA)
Comprehensive visual and statistical analysis of:
1. Category class distribution
2. Document word length distributions (raw vs cleaned)
3. Top 30 most frequent words
4. Global WordCloud
5. N-Grams (Unigrams, Bigrams, Trigrams)
6. Class-specific distinctive keywords


In [ ]:
# 1. Class Distribution
plt.figure(figsize=(12, 6))
cat_counts = df_raw['category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, color='#1E3A8A')
plt.title("Resume Distribution across 24 Job Categories", fontsize=14, fontweight='bold')
plt.xlabel("Number of Resumes")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

# 2. Text Length Distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_raw['word_count'], bins=40, kde=True, ax=ax[0], color='#2563EB')
ax[0].set_title("Word Count Distribution", fontweight='bold')
ax[0].axvline(df_raw['word_count'].median(), color='red', linestyle='--', label=f'Median: {df_raw["word_count"].median():.0f}')
ax[0].legend()

sns.boxplot(x=df_raw['word_count'], ax=ax[1], color='#93C5FD')
ax[1].set_title("Word Count Boxplot (Outlier Detection)", fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Display Distinctive Class Keywords
with open('../data/processed/eda_plots/class_distinctive_terms.json') as f:
    class_terms = json.load(f)

print("Sample Class-Specific Distinctive Terms (TF-IDF):")
for cat in ['INFORMATION-TECHNOLOGY', 'FINANCE', 'HEALTHCARE', 'CHEF', 'ADVOCATE', 'AVIATION']:
    print(f"  {cat:24s}: {', '.join(class_terms[cat][:6])}")


## Stage 5 ? Domain-Aware Preprocessing
Resume text cleaning must **never blindly delete punctuation**, because critical tech skills such as C++, .NET, C#, and compound terms would otherwise be destroyed.
Our preprocessor:
- Normalizes whitespace, tabs, and line breaks
- Cleans HTML markup and encoding artifacts
- Normalizes URLs and emails
- **Protects technical tokens** (C++ ? cplusplus, .NET ? dotnet, C# ? csharp, Python, SQL, AWS, etc.)
- Normalizes case and removes remaining noise
- Applies stopword removal and lemmatization


In [ ]:
test_snippet = "Experienced Software Architect skilled in C++, .NET, C#, Python, SQL, AWS, and NLP. Portfolio: https://github.com/myrepo. Contact: engineer@domain.com."
cleaned_snippet = clean_resume_text(test_snippet)
detected_skills = extract_skills_from_text(test_snippet)

print("--- Preprocessing Verification ---")
print("Raw Input:        ", test_snippet)
print("Cleaned Output:   ", cleaned_snippet)
print("Detected Skills:  ", detected_skills)

# Assertions verifying protection
assert 'cplusplus' in cleaned_snippet, "C++ lost!"
assert 'dotnet' in cleaned_snippet, ".NET lost!"
assert 'csharp' in cleaned_snippet, "C# lost!"
assert 'python' in cleaned_snippet, "Python lost!"
assert 'sql' in cleaned_snippet, "SQL lost!"
assert 'aws' in cleaned_snippet, "AWS lost!"
assert 'nlp' in cleaned_snippet, "NLP lost!"
print("\n? All technical keyword preservation tests passed successfully!")


## Stage 6 ? Classical Machine Learning Baselines
We perform a **Stratified 70/15/15 Split** (Train / Validation / Test) with 
andom_state=42.
The TF-IDF vectorizer is fitted **strictly on the training split**, completely avoiding data leakage.
We benchmark:
1. **Multinomial Naive Bayes**
2. **Logistic Regression (balanced)**
3. **Linear Support Vector Classifier (calibrated)**


In [ ]:
cleaned_csv = os.path.join('..', 'data', 'processed', 'cleaned_resumes.csv')
df_clean = pd.read_csv(cleaned_csv)

split_dict = split_data(df_clean, text_col='cleaned_text', label_col='category', train_size=0.70, val_size=0.15, test_size=0.15, random_state=42)
X_train, y_train = split_dict['X_train'], split_dict['y_train']
X_val, y_val = split_dict['X_val'], split_dict['y_val']
X_test, y_test = split_dict['X_test'], split_dict['y_test']
label_encoder = split_dict['label_encoder']
class_names = list(label_encoder.classes_)

# Fit TF-IDF only on training data
X_train_vec, X_val_vec, X_test_vec, tfidf_vec = build_tfidf_features(
    X_train, X_val, X_test,
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2,
    max_df=0.75,
    sublinear_tf=True
)

classical_models, classical_metrics, classical_preds, classical_probs = train_classical_models(
    X_train_vec, y_train, X_val_vec, y_val, X_test_vec, y_test
)


## Stage 7 ? Deep Learning Architecture (Word2Vec + BiLSTM)
We train continuous skip-gram **Word2Vec embeddings** (150-dim) on the training corpus.
A **2-layer Bidirectional LSTM** with dual pooling (Global Max Pooling + Average Pooling) and class-weighted CrossEntropyLoss is trained to predict the 24 categories.


In [ ]:
# Load trained Word2Vec and PyTorch model
w2v_model = train_word2vec_embeddings(list(X_train), vector_size=150, window=5, min_count=2, epochs=12)
embedding_matrix, word2idx = build_embedding_matrix(w2v_model, embedding_dim=150)

X_train_seq = texts_to_sequences(list(X_train), word2idx, max_len=300)
X_val_seq = texts_to_sequences(list(X_val), word2idx, max_len=300)
X_test_seq = texts_to_sequences(list(X_test), word2idx, max_len=300)

dl_model, dl_metrics, dl_preds, dl_probs = train_dl_model(
    X_train_seq, y_train,
    X_val_seq, y_val,
    X_test_seq, y_test,
    embedding_matrix=embedding_matrix,
    num_classes=len(class_names),
    batch_size=64,
    epochs=10
)


## Stage 8 ? Model Benchmark & Evaluation
We compare all models on the held-out, unseen test set across **Accuracy, Macro-F1, Weighted-F1, Precision, and Recall**.


In [ ]:
all_results = dict(classical_metrics)
all_results['Word2Vec + BiLSTM'] = dl_metrics
df_comp = generate_comparison_table(all_results, save_path='../models/model_comparison.csv')
df_comp


In [ ]:
# Plot Confusion Matrix for Best Model
best_model_name = df_comp.iloc[0]['model']
best_preds = dl_preds if 'BiLSTM' in best_model_name else classical_preds[best_model_name]
plot_confusion_matrix(y_test, best_preds, class_names, best_model_name, save_path='../models/confusion_matrix.png')

# Display Confusion Matrix image
from IPython.display import Image
Image('../models/confusion_matrix.png')


## Stage 9 ? Error Analysis & Root Cause Diagnosis
We systematically inspect misclassified resumes on the held-out test set to identify why the model erred.


In [ ]:
err_df = perform_error_analysis(
    list(X_test), y_test, dl_preds, dl_probs, class_names, save_path='../models/error_analysis.csv'
)
print(f"Error Rate: {len(err_df)} / {len(y_test)} ({len(err_df)/len(y_test)*100:.2f}%)")
print("\nError Category Breakdown:")
print(err_df['primary_reason'].value_counts())
err_df[['actual_category', 'predicted_category', 'confidence', 'primary_reason']].head(10)


## Stage 10 ? Final Prediction Pipeline & Real Resume Demo
We demonstrate the unified ResumeClassifierPipeline on raw text and direct PDF inputs.


In [ ]:
pipeline = ResumeClassifierPipeline(models_dir='../models')

# Test 1: Real PDF Resume
pdf_sample = '../data/raw/AUTOMOBILE/78357954.pdf'
pdf_res = pipeline.predict(pdf_sample, is_pdf=True, model_type='dl')
print(f"=== Real PDF Prediction ===")
print(f"File:       {pdf_sample}")
print(f"Predicted:  {pdf_res['predicted_category']} ({pdf_res['confidence']}%)")
print(f"Top 3:      {pdf_res['top_3_predictions']}")
print(f"Skills:     {pdf_res['extracted_skills']}")

# Test 2: Unseen Candidate Text
unseen_text = """
Senior Healthcare Administrator with 8 years overseeing clinical operations,
patient record confidentiality, HIPAA compliance, hospital staff scheduling, and medical billing.
"""
txt_res = pipeline.predict(unseen_text, is_pdf=False, model_type='dl')
print(f"\n=== Unseen Candidate Text Prediction ===")
print(f"Predicted:  {txt_res['predicted_category']} ({txt_res['confidence']}%)")
print(f"Top 3:      {txt_res['top_3_predictions']}")
